# 02-lazy-evaluation

02-lazy-evaluation/01-lazy-evaluation-and-dag.py
Demonstrates lazy evaluation, DAG construction, Catalyst optimizer, and caching.

Patterns covered:
  1.  Transformation vs Action (lazy vs eager)
  2.  Transformation chain — execution triggered only by action
  3.  explain(True) — all four plan levels
  4.  DAG: jobs → stages → tasks, separated by shuffles
  5.  Wide vs narrow transformations
  6.  Lineage via rdd.toDebugString
  7.  Catalyst optimizer — predicate pushdown effect on join plans
  8.  Action types with timing: count / first / take / collect
  9.  cache() effect — first action materializes; second reads from cache
  10. show() vs collect() safety warning

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views

import time
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("02-lazy-evaluation")
dfs   = register_views(spark)
emp   = dfs["employees"]

## 1. Transformation vs Action

In [ ]:
print("=" * 60)
print("1. Transformation vs Action")
print("=" * 60)
print("""
   TRANSFORMATIONS (lazy — build the DAG plan, no execution):
     filter, select, withColumn, join, groupBy, orderBy,
     repartition, union, distinct, withColumnRenamed, drop

   ACTIONS (eager — trigger a Spark job / execution):
     show(), count(), collect(), take(n), first(),
     write.save(), foreach(), toPandas()

   Rule: Spark does nothing until an action is called.
   Every action compiles the full DAG and submits a job to the scheduler.
""")

## 2. Transformation chain — no execution until action

In [ ]:
print("=" * 60)
print("2. Transformation chain — execution deferred until action")
print("=" * 60)

step1 = emp.filter(F.col("status") == "Active")
step2 = step1.select("emp_id", "first_name", "last_name", "salary", "dept_id")
step3 = step2.withColumn(
    "salary_band",
    F.when(F.col("salary") > 150000, "Senior").otherwise("Other")
)

# Nothing has executed yet — all three lines above built query plan nodes only.
print("   Transformation chain built. Triggering action...")
step3.show(5)  # execution happens here — Spark submits a job

## 3. explain(True) — parsed / analyzed / optimized / physical plans

In [ ]:
print("=" * 60)
print("3. explain(True) — all four plan levels for a filtered query")
print("=" * 60)
print("   Query: emp.filter(dept_id == 1).select(emp_id, salary)")
print("-" * 60)
emp.filter(F.col("dept_id") == 1).select("emp_id", "salary").explain(True)
# Output sections:
#   == Parsed Logical Plan ==     (raw AST from the query)
#   == Analyzed Logical Plan ==   (after resolution / type checking)
#   == Optimized Logical Plan ==  (after Catalyst rule passes)
#   == Physical Plan ==           (execution strategy)

## 4. DAG — jobs, stages, tasks

In [ ]:
print("=" * 60)
print("4. DAG — jobs → stages → tasks")
print("=" * 60)
print("""
   Each ACTION creates one Spark JOB (visible in Spark UI → Jobs tab).
   Each JOB is divided into STAGES.
   Stages are separated by SHUFFLES (wide transformations that
   require data redistribution across partitions, e.g. groupBy+agg, join).
   Each stage runs as many TASKS as there are partitions.

   Narrow transformations (filter, select, withColumn, union) execute
   within a single stage — no shuffle, no stage boundary.

   Wide transformations (groupBy+agg, join, repartition, distinct, orderBy)
   force a stage boundary: Spark writes shuffle files and readers pick them up
   in the next stage.
""")

## 5. Wide vs narrow transformations — explain output comparison

In [ ]:
print("=" * 60)
print("5. Wide vs narrow transformations")
print("=" * 60)

print("\n--- Narrow: filter (no Exchange / shuffle node expected) ---")
emp.filter(F.col("dept_id") == 1).explain()

print("\n--- Wide: groupBy+count (Exchange node visible = shuffle) ---")
emp.groupBy("dept_id").count().explain()

## 6. Lineage — rdd.toDebugString

In [ ]:
print("=" * 60)
print("6. Lineage via rdd.toDebugString")
print("=" * 60)
lineage_bytes = emp.filter(F.col("status") == "Active").rdd.toDebugString()
print(lineage_bytes.decode())

## 7. Catalyst optimizer — predicate pushdown and join reordering

In [ ]:
print("=" * 60)
print("7. Catalyst optimizer — predicate pushdown")
print("=" * 60)
print("""
   Catalyst rewrites logical plans automatically:
     - Predicate pushdown: filters move as close to the data source as possible
       so that fewer rows are read / shuffled.
     - Constant folding, column pruning, join reordering.

   Below we compare two equivalent queries.
   Catalyst typically pushes both to the same physical plan (AQE may differ).
""")

print("\n--- Without manual pushdown: filter AFTER join ---")
joined_after = emp.join(dfs["departments"], "dept_id", "left")
joined_after.filter(F.col("dept_id") == 1).explain()

print("\n--- With manual pushdown: filter BEFORE join ---")
emp.filter(F.col("dept_id") == 1).join(dfs["departments"], "dept_id", "left").explain()

# Both plans are often identical because Catalyst's PushDownPredicate rule
# automatically moves the filter before the join in the first case.

## 8. Action types with timing

In [ ]:
print("=" * 60)
print("8. Action types — timing")
print("=" * 60)

actions = [
    ("count",    lambda df: df.count()),
    ("first",    lambda df: df.first()),
    ("take(3)",  lambda df: df.take(3)),
    ("collect",  lambda df: df.collect()),
]
for action_name, action_fn in actions:
    t0     = time.time()
    result = action_fn(emp)
    elapsed = time.time() - t0
    # Show a condensed result representation
    if isinstance(result, list):
        summary = f"[{len(result)} rows]"
    else:
        summary = str(result)[:60]
    print(f"   {action_name:<10s}  {elapsed:.3f}s   result={summary}")

print("""
   Notes:
     count()   — scans all partitions, returns a single Long.
     first()   — returns the first Row; may short-circuit.
     take(n)   — returns n Rows to driver; optimized to scan minimally.
     collect() — returns ALL rows to driver. DANGEROUS on large data.
""")

## 9. cache() — first action materializes; second action reads from cache

In [ ]:
print("=" * 60)
print("9. cache() effect")
print("=" * 60)

emp_cached = emp.cache()   # registers the intent to cache — no execution yet

t0 = time.time()
cnt1 = emp_cached.count()  # materializes the cache (check Spark UI → Storage)
t1 = time.time() - t0
print(f"   First count()  (materializes cache): {cnt1} rows in {t1:.3f}s")

t0 = time.time()
cnt2 = emp_cached.count()  # reads from in-memory cache
t2 = time.time() - t0
print(f"   Second count() (from cache)        : {cnt2} rows in {t2:.3f}s")
print(f"   Speedup: {t1/max(t2,0.001):.1f}x  (check Spark UI Storage tab for cache details)")

emp_cached.unpersist()
print("   emp unpersisted.")

## 10. show() vs collect() — safety warning

In [ ]:
print("=" * 60)
print("10. show() vs collect()")
print("=" * 60)
print("""
   show(n)    — streams n rows to the console; SAFE on any size DataFrame.
                Spark only materializes the first n rows from each partition.

   collect()  — pulls ALL rows from all partitions into driver memory.
                Safe ONLY when DataFrame is small (e.g., after heavy aggregation).
                On a 100M-row table this will OOM or freeze the driver.

   Best practice:
     Use show() for inspection during development.
     Use collect() only on known-small DataFrames (e.g., after .count() confirms).
     Use toPandas() with the same caution as collect().
""")

stop_and_wait(spark)